# Import des librairies

In [250]:
from dataclasses import dataclass
import re
from itertools import permutations
from collections import defaultdict

# Definition classe

In [251]:
@dataclass
class Neighbors():
    neighbor1: str
    neighbor2: str
    global_happiness: int

# Lecture du fichier source

In [252]:
pattern = r"(gain|lose) (\d+)"

with open('real.txt') as test:
    file = test.read().split('\n')
file

['Alice would lose 57 happiness units by sitting next to Bob.',
 'Alice would lose 62 happiness units by sitting next to Carol.',
 'Alice would lose 75 happiness units by sitting next to David.',
 'Alice would gain 71 happiness units by sitting next to Eric.',
 'Alice would lose 22 happiness units by sitting next to Frank.',
 'Alice would lose 23 happiness units by sitting next to George.',
 'Alice would lose 76 happiness units by sitting next to Mallory.',
 'Bob would lose 14 happiness units by sitting next to Alice.',
 'Bob would gain 48 happiness units by sitting next to Carol.',
 'Bob would gain 89 happiness units by sitting next to David.',
 'Bob would gain 86 happiness units by sitting next to Eric.',
 'Bob would lose 2 happiness units by sitting next to Frank.',
 'Bob would gain 27 happiness units by sitting next to George.',
 'Bob would gain 19 happiness units by sitting next to Mallory.',
 'Carol would gain 37 happiness units by sitting next to Alice.',
 'Carol would gain 45 h

# Nettoyage

In [253]:
def func_decode_line_file(in_line: str) -> dict:

    # Split la phrase en une liste de mots
    line_splited = in_line.split()
    
    personne1 = line_splited[0]
    personne2 = line_splited[-1][:-1]


    resultat = re.search(pattern, in_line)
    type_happiness, score_happiness_str = resultat.groups()

    score_happiness = int(score_happiness_str)

    if type_happiness == 'lose':
        score_happiness *= -1

    
    return (personne1, personne2, score_happiness)

# Construit un dictionnaire d'affinité

In [254]:
dict_total_affity_two_persons = defaultdict(dict)
# Pour chaque ligne du fichier source
for line in file:
    
    personne1, personne2, score_happiness = func_decode_line_file(line)

    dict_total_affity_two_persons[personne1][personne2] = score_happiness

dict_total_affity_two_persons

defaultdict(dict,
            {'Alice': {'Bob': -57,
              'Carol': -62,
              'David': -75,
              'Eric': 71,
              'Frank': -22,
              'George': -23,
              'Mallory': -76},
             'Bob': {'Alice': -14,
              'Carol': 48,
              'David': 89,
              'Eric': 86,
              'Frank': -2,
              'George': 27,
              'Mallory': 19},
             'Carol': {'Alice': 37,
              'Bob': 45,
              'David': 24,
              'Eric': 5,
              'Frank': -68,
              'George': -25,
              'Mallory': 30},
             'David': {'Alice': -51,
              'Bob': 34,
              'Carol': 99,
              'Eric': 91,
              'Frank': -38,
              'George': 60,
              'Mallory': -63},
             'Eric': {'Alice': 23,
              'Bob': -69,
              'Carol': -33,
              'David': -47,
              'Frank': 75,
              'George': 82,
   

# On crée un set de personnes

In [255]:
set_personnes = set(dict_total_affity_two_persons.keys())
set_personnes

{'Alice', 'Bob', 'Carol', 'David', 'Eric', 'Frank', 'George', 'Mallory'}

In [256]:
def func_calcul_mutual_affinity(
    in_personne1: str,
    in_personne2: str, 
    in_dict_total_affity_two_persons: dict[str]
    ) -> int:

    # Récupère l'affinité de la personne 1 ---> la personne 2
    affinity_p1_to_p2 = in_dict_total_affity_two_persons[in_personne1][in_personne2]

    # Récupère l'affinité de la  personne 2 ---> la personne 1
    affinity_p2_to_p1 = in_dict_total_affity_two_persons[in_personne2][in_personne1]

    # Calcule l'affinité mutuelle 
    mutual_affinity = affinity_p1_to_p2 + affinity_p2_to_p1

    return mutual_affinity

In [257]:
def func_calcul_affinity_disposition(
    in_disposition: tuple[str],
    in_dict_total_affity_two_persons: dict[str]
):
    
    # Initialise l'affinité de la disposition
    disposition_affinity = 0

    # Pour chaque paire de personnes de la disposition
    for personne1, personne2 in zip(in_disposition,in_disposition[1:]+in_disposition[:1]):
        
        # On récupère leur affinité
        affinity = func_calcul_mutual_affinity(
            in_personne1 = personne1,
            in_personne2 = personne2,
            in_dict_total_affity_two_persons = in_dict_total_affity_two_persons
        )

        # On ajoute leur affinité à la somme des affinités pour cette disposition
        disposition_affinity += affinity

    return disposition_affinity

In [ ]:
# Trouve le meilleure score d'affinité parmis toutes les dispositions possibles
max_affinity_disposition = max(
    func_calcul_affinity_disposition(
        in_disposition = disposition,
        in_dict_total_affity_two_persons = dict_total_affity_two_persons
    )
    for disposition 
    in permutations(set_personnes)
)

max_affinity_disposition

618